# **Goal**

The purpose of this notebook is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

###**Pythia Model Family**

- Pythia models were originally designed for interpretability research.

- All Pythia models were trained on the same data in the same order.

- Training data: All Pythia models are trained on The Pile. The Pile is a 825GiB general-purpose dataset in English. It contains texts from 22 diverse sources, roughly broken down into five categories: academic writing (e.g. arXiv), internet (e.g. CommonCrawl), prose (e.g. Project Gutenberg), dialogue (e.g. YouTube subtitles), and miscellaneous (e.g. GitHub, Enron Emails).

- Each model saw 299,892,736,000 ~= 300B tokens during training.

####**Pythia 160M**

- <u>Total params</u>: 162,322,944. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 85,056,000. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 12
- <u>Model dimension</u>: 768. I.e.: Every token is represented internally by a vector of length 768.
- <u>MLP units per layer:</u> 3072
- <u>Heads</u>: 12
- <u>Vocabulary size:</u> 50,304. Amount of tokens in vocabulary.
- <u>Max context window</u>: 2048 tokens.

Load model

In [ ]:
from transformers import GPTNeoXForCausalLM, AutoTokenizer

model = GPTNeoXForCausalLM.from_pretrained(
  "EleutherAI/pythia-160m-deduped",
  revision="step143000",
  cache_dir="./pythia-160m-deduped/step143000",
)

tokenizer = AutoTokenizer.from_pretrained(
  "EleutherAI/pythia-160m-deduped",
  revision="step143000",
  cache_dir="./pythia-160m-deduped/step143000",
)

inputs = tokenizer("Hello, I am", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  649MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'Hello, I am a newbie to the web and I have been trying to get my head around the concept of a'

Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [ ]:
inputs = tokenizer("How much do you like ice cream?", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=28) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'How much do you like ice cream?\n\nI love ice cream. I love the ice cream. I love the ice cream. I'

In [ ]:
inputs = tokenizer("The capital of France is", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=25) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'The capital of France is the city of Paris, and the capital of the world.\n\nThe French capital is the capital'

In [ ]:

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(output[0])
print(len(output[0]))
print()
print(tokenizer.decode(output[0]))

tensor([  510,  5347,   273,  6181,   310,   275,   775, 11441,    14, 17483,
        31453,   860,    13,   253,  6253,  2846,   275,   253,  2919,   285,
          253,   954,  4633, 22777, 12095,    13,   342,  2761,  7469,  3041,
        12942,   247,   807,    15,   187,   187,   510,  2846,   556, 12103,
        44373,   275,   253,  2469,  1643,  1107,    13,   342,   247,  1534,
         1180,   273, 24359, 13975,   253])
55

The capital of France is in Champs-Élysé, the largest city in the region and the most popular tourist destination, with almost 300 million visitors a year.

The city has gained prominence in the past few years, with a significant number of tourists visiting the


In [ ]:
inputs = tokenizer("What is love?", return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

Inputs:tensor([1276,  310, 2389,   32])
Outputs: tensor([10540,   476,   320,  1119,   275,   253,  1039,   352,  9727,   368,
           15,   187,    29,    51,  2066,    31,  3780,   871,  2139,   309,
         2550,  1014,  4684,   281,   619,  4382,  3066,  8573, 34977,   326,
          434,   849,   891,  1353,  2820,     2,   667,  5697,  8220,  1310,
          627,   403,  2571,    13,  4496,  1339,   479,  1361,   342,   436])

What is love? Love can be found in the way it moves you.
<Ruff> anyone know why I cannot even connect to my computer via internet :( that's how i'm trying! any ideas?? If there are others, please let me help with this


Here we are just printing the internal shape of one layer of the model.

In [ ]:
print(model.gpt_neox.layers[0].mlp)

GPTNeoXMLP(
  (dense_h_to_4h): Linear(in_features=768, out_features=3072, bias=True)
  (dense_4h_to_h): Linear(in_features=3072, out_features=768, bias=True)
  (act): GELUActivation()
)


In [ ]:
model.config.intermediate_size

3072

The input vector representation is 768-dimensional. The number of MLP units in the layer is 3072. Each of these produces a scalar from the input through learned weights and bises, so the layer produces a transformed and expanded 3072-d vector. After activation using GELU (Gaussian Error Linear Unit), these values are transformed back into a new 768-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{GELU}(x) = x \cdot \Phi(x) = 0.5x \left(1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right)$, where $Φ(x)$ is the cumulative distribution function of the standard normal distribution, and $\text{erf}(x)$ is the Gauss error function.

**MLP Unit Ablation Experiments**

In [ ]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # Load weights in layer layer_idx
    mlp = model.gpt_neox.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    # Indexes in [0, 3071]
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    #
    def ablation_hook(module, inputs, output):
        indices_device = indices.to(output.device)
        output = output.clone()
        output[..., indices_device] = 0
        return output

    handle = mlp.act.register_forward_hook(ablation_hook)

    return handle, indices, layer_idx

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [ ]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,  2846,    13,   285,   697,  3072,   556,   644,  3629,  1580,
         1283,  2941,    15,   187,   688,  1722,  2511,   352,   369,   247,
         1659,   835,  1142,   952,   497,  3811,   275, 14192,    28,  3063,
          627,   403,   625,   685,   337,  3041, 21251,   665,  3153,  2708,
          616,  2097,    27,   597,   452,   642, 16347,   390,  6152,  8364])

Zeroed activations in 0.0% of units in layer 0
The capital of France is the city, and its population has been increasing since 1803.
In 1789 it was a place where many people were living in poverty; today there are more than 1 million inhabitants who live below their means: they have no savings or credit cards


$25\%$ of first layer

In [ ]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,   954,  1774,   285,    13,   187,  1171,   512,    15,   380,
         5112,   403,   247,  5674,   326,   556,   644,   275,  6242,   323,
          625,   685,   767, 14020,    28,   352,   369, 11420,   407, 32436,
         6490,   327,  2552,   337,   296,  1722,  4590,   313,   395,   534,
          344,   574,  2168,  1160,   521,   806,  3177,   387, 14275,   432])

Zeroed activations in 25.0% of units in layer 0
The capital of France is the most important and,
of all. The French are a nation that has been in existence for more than two centuries; it was founded by Napoleon III on May 1st 1793 (and which he had already made his first attempt at independence from


$50\%$ of first layer

In [ ]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([ 253,  954, 1774,  285,   13,  275,  512, 2892,   15,  187,  380, 5112,
        1342,  369,  247,  637,  665,  556,  644,  271, 9558,  326,  344,  434,
         417,  581,  281,  320,  521, 1445,  390,  352,  347,  973,  323,  436,
         309,  452,  594, 1199,  625,  685,  368,  403,  619, 1495,  533,  752,
         476,  359])

Zeroed activations in 50.0% of units in layer 0
The capital of France is the most important and, in all history.
 The Frenchman was a man who has been an artist that he's not one to be his art or it as well for this I have so much more than you are my life but what can we


$75\%$ of first layer

In [ ]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  187,    13,    15,    28,     3,   407,    14,    27,    16,   285,
            2,   275,   253,  1051,   281,   432,   247, 50276,   186, 50275,
        50274,   209,   623, 28910, 50272,    29, 50273,   996, 50269,   551,
        50271,    31, 50267,    33,  2490,  1040,   313,   535,  1214,   387,
          544,   342,   327,    62,   340,   745,  3706,  6781,  4725,   748])

Zeroed activations in 75.0% of units in layer 0
The capital of France is
,.;" by-:/ and! in the... to from a  	        		 	      <     
	          {       >           @ 
 | (

 @ at [ with on] y off ; --> /> }


$100\%$ of first layer

In [ ]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  187,   423,    13,    15,    27,    28,     2,   407,    14,   544,
          367,   275,   268,   591,  3402,  2245,  2177,  4980,   327,  7503,
         1612,    62,   362,   295,  1708,   285,    16,  4376,   246,  6152,
          278,  9176,  1386, 12275,   941,  1048,  3148,  2978,  1979,  1755,
          418,   673,   186,   996,   313,     3,    10,    31,    60,   535])

Zeroed activations in 100.0% of units in layer 0
The capital of France is
end,.:;! by- [ P in p per wall port vi modern on virtual power] v n light and/ price t credit m pin line pixel data long display length size top L time	
	 (")>[




**Attention Head Ablation Experiments**

First let's check the attention heads

In [ ]:
print(model.gpt_neox.layers[0].attention)

GPTNeoXAttention(
  (query_key_value): Linear(in_features=768, out_features=2304, bias=True)
  (dense): Linear(in_features=768, out_features=768, bias=True)
)


We see that the attention mechanism expects a 768d input (the token embedding or output from the previous block), which gets turned into a 2304-d vector. This corresponds to calculating its query, key, and value vectors and concatenating these. There are 12 heads computing the attention mechanisms between QKV in slices of 64d vectors.

These attention contributions from each head are then concatenated to form another 768d vector, which in turn gets transformed into the final attention contribution vector that combines and mixes the computations from all heads, and is incorporated into the input for the next transformer block.

In [ ]:
print(model.gpt_neox.layers[0].attention.dense)

Linear(in_features=768, out_features=768, bias=True)


In [ ]:
model.config.hidden_size

768

In [ ]:
model.config.num_attention_heads

12

In [ ]:
model.config.hidden_size // model.config.num_attention_heads

64

Attention head ablation function

In [ ]:
def add_head_ablation(model, head_indices, layer_idx):

    num_layers = model.config.num_hidden_layers
    #Embedding dimension
    hidden = model.config.hidden_size
    #Number of attention heads
    num_heads = model.config.num_attention_heads
    #Projection dimension per attention head
    head_dim = hidden // num_heads

    if not 0 <= layer_idx < num_layers:
        raise ValueError(f"layer_idx must be between 0 and {num_layers - 1}")

    # Load final attention module
    attention_dense = model.gpt_neox.layers[layer_idx].attention.dense

    def head_ablation_hook(module, inputs):
        x = inputs[0].clone()

        for head_idx in head_indices:
            if not 0 <= head_idx < num_heads:
                raise ValueError(f"head_idx must be between 0 and {num_heads - 1}")
            #compute indices corresponding to specific head
            start = head_idx * head_dim
            end = start + head_dim

            #Select indices in the head outputs vector corresponding to chosen head and zero them
            x[..., start:end] = 0

        return (x,) + inputs[1:]

    # Note this is a pre-hook because we want to intervene components before the forward pass.
    handle = attention_dense.register_forward_pre_hook(head_ablation_hook)

    return handle, head_indices, layer_idx

Here we try zeroing the outputs of different numbers of heads of the first (0th) layer.

Remove 1 head from layer 0

In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of Paris is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 7785,  310])
Outputs: tensor([  253,  2846,    15,   187,   688,  1283,  2090,    13,   352,   369,
        27624,   281,   346, 36062,     3,   275, 10390,   285, 19248,   272,
          697,  7815,   665,   452,   644,  5339,   407,  5112,  9647,  1309,
         3645,  3660,   309,    28,   436,  1416,   556,  1580,  2489, 39882,
          342,   247,  1180,   390,   625,   685,   581,  4289,  8014,   952])

Zeroed head [0] in layer 0
The capital of Paris is the city.
In 1869, it was renamed to "Paris" in honor and honouring its citizens who have been killed by French soldiers during World War I; this name has since become synonymous with a number or more than one hundred thousand people


Ablate 3 attention heads in layer 0

---



In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of Paris is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 7785,  310])
Outputs: tensor([  253,  2846,   835,   352,   556,   247,  1180,    15,   187,   688,
         1635,   281,   436,    13,   627,   403,   671,  2067,   643,  8238,
          275,  6181,    27,   535, 50276,    11,   378,  1120,   313,   249,
         5112,    10,  1108,   380,   954,  8530,  1659,   323,   270,  3370,
         2196,   285,  8965,    28,   368,   476,   923,  1142,  5053,   751])

Zeroed head [0, 1, 2] in layer 0
The capital of Paris is the city where it has a number.
In addition to this, there are also several other cities in France:

  * Brest (in French) – The most famous place for billiards and bars; you can see many places like


Ablate 6 attention heads in layer 0

In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of Paris is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 7785,  310])
Outputs: tensor([  253,  2846,   326,   368,   403,  1469,   281,   320,   275,   247,
         1659,   835,   627,   434,   642,   581,    15,   187,    42,  1353,
          417,  2119,   604,   309,   476,   513,   352,    13,   533,   436,
          588,   789,   323,   479,   285, 18670,  1056,   619,  1495,  6927,
            2,    50,    27,  1737,  1057,   346,   783,     3,  1599,    32])

Zeroed head [0, 1, 2, 3, 4, 5] in layer 0
The capital of Paris is the city that you are going to be in a place where there's no one.
I'm not sure if I can do it, but this will work for me and hopefully make my life easier!Q: What does "the" mean?


Ablate 9 attention heads in layer 0

In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5,7,8,9],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of Paris is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 7785,  310])
Outputs: tensor([  253,  2846,   835,   352,   434,  4441,    15,   187,   783,   954,
         1774,  2181,   275,  1495,    13,   285,   326,   368,   476,   626,
          320,   247,   629,    14,  2606,  8183,   390,   271,  4184,   387,
          667,  2567,    28,   533,   604,   634,  2628,  4419,   281,   789,
          323,   625,   685,   581,   807,   840,   627,   403,   642, 38219])

Zeroed head [0, 1, 2, 3, 4, 5, 7, 8, 9] in layer 0
The capital of Paris is the city where it's located.
the most important thing in life, and that you can't be a part-time employee or an intern at any company; but if your job requires to work for more than one year then there are no excuses


Ablate 12 attention heads in layer 0

In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5,6,7,8,9,10,11],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of Paris is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 7785,  310])
Outputs: tensor([  187, 36062,    13,  6181,    15,   380,   806,  1984,   407,   247,
         5112,  8406,   281,   320,  3863,   275,   253,  1986,  2077,  1342,
          369,  3542,   323,   521,  4475,   285,  3347,    14,   249,  6937,
          387,  1728,    28,   344,   574,   644,  4028,   670,   779,  1580,
         1078,  1283,  1107,  1711,   672,   597,   497,  7028,   313,   395])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] in layer 0
The capital of Paris is
Paris, France. The first book by a French writer to be published in the United Statesman was written for his wife and son-inlaw at home; he had been writing about him since before 18 years old when they were married (and


Ablate 12 attention heads in layer 11

In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5,6,7,8,9,10,11],
    layer_idx=11,
)

inputs = tokenizer(
    '''The capital of Paris is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 7785,  310])
Outputs: tensor([  253,  2846,   434,  6253,    13,   342,   247,  3072,   326,   556,
          644,  5675, 25060,   689,  3332,  1107,    15,   187,   688,  1635,
          281,   697,  6793,  8928,   285,  9493, 31154,   824,   347, 37702,
           14,    37,   482,   372,   657,  4002,   313,   783, 17172,  6105,
          275,  6181,   582,   352,   671, 38716,   271, 13943,  4849,   323])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] in layer 11
The capital of Paris is the city's largest, with a population that has been growing steadily over recent years.
In addition to its rich cultural and historical attractions such as Notre-Dame de Ville (the oldest church in France), it also boasts an impressive collection for
